<a href="https://colab.research.google.com/github/FSA-1606/chat-voz-openai-colab-DIO/blob/main/Desafio_Voz_ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [84]:
from openai import OpenAI

client = OpenAI(
    api_key= "COLOQUE SUA CHAVE AQUI"
)

language = "pt"   #
chat_history = [] # memória do chat

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Responda apenas: OK"
)

print(response.output_text)


OK


In [86]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=10):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

In [70]:
!pip install git+https://github.com/openai/whisper.git -q


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [71]:
import whisper

model = whisper.load_model("small")

def transcribe_audio(audio_path):
    result = model.transcribe(audio_path, fp16=False, language=language)
    return result["text"]


In [67]:
def ask_chatgpt(user_text):
    global chat_history

    # adiciona fala do usuário
    chat_history.append({
        "role": "user",
        "content": user_text
    })

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=chat_history
    )

    assistant_reply = response.output_text

    # adiciona resposta do bot à memória
    chat_history.append({
        "role": "assistant",
        "content": assistant_reply
    })

    return assistant_reply


In [72]:
from gtts import gTTS
from IPython.display import Audio, display

def speak(text):
    tts = gTTS(text=text, lang=language, slow=False)
    file = "/content/response.wav"
    tts.save(file)
    display(Audio(file, autoplay=True))


In [88]:
# Transcreve o áudio gravado
result = model.transcribe(
    record_file,
    fp16=False,
    language=language
)

transcription = result["text"]
print("Você:", transcription)


Você:  Eu...


In [89]:
exit_words = ["sair", "encerrar", "finalizar", "tchau"]

while True:
    print("🎤 Ouvindo...")
    audio_file = record(sec=5)

    user_text = transcribe_audio(audio_file).strip()
    print("👤 Você:", user_text)

    if not user_text:
        print("⚠️ Nenhuma fala detectada.\n")
        continue

    # ENCERRAR CONVERSA
    if any(word in user_text.lower() for word in exit_words):
        print("👋 Conversa encerrada.")
        speak("Conversa encerrada. Até mais!")
        break

    reply = ask_chatgpt(user_text)
    print("🤖 ChatGPT:", reply)
    speak(reply)


🎤 Ouvindo...


<IPython.core.display.Javascript object>

👤 Você: Cola a soma de 5 mais 5.
🤖 ChatGPT: Claro! A soma de 5 mais 5 é:

5 + 5 = 10

Se precisar de mais ajuda, é só pedir!


🎤 Ouvindo...


<IPython.core.display.Javascript object>

...

🎤 Ouvindo...


<IPython.core.display.Javascript object>

👤 Você: 
⚠️ Nenhuma fala detectada.

🎤 Ouvindo...


<IPython.core.display.Javascript object>

👤 Você: Sair!
👋 Conversa encerrada.
